# Regularized Regression Tuning

## 1. Imports

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. Load processed data

In [2]:
df = pd.read_csv("../data/processed/member_analysis_ready.csv")

print(df.shape)
df.head()



(3000, 25)


,member_id,age,gender,region,plan_type,sdoh_risk_score,chronic_condition_count,engagement_score,pcp_attributed_24mo,prior_awv_count,...,engagement_group,age_group,high_cost_member,chronic_burden_group,sdoh_risk_group,prior_awv_group,total_acute_visits,acute_utilization_group,pcp_status,has_acute_utilization
0,M00001,69,Male,Suburban,Medicare Advantage,40.7,2,68.9,1,1,...,Q4_High,65-79,1,Moderate,Q1_Low,1,2,Low,Attributed,1
1,M00002,32,Female,Urban,DSNP,80.0,3,30.4,0,0,...,Q1_Low,18-34,0,Moderate,Q4_High,0,2,Low,Not Attributed,1
2,M00003,89,Female,Suburban,Medicare Advantage,49.6,3,86.3,1,3,...,Q4_High,80+,0,Moderate,Q2,3,0,NaN,Attributed,0
3,M00004,78,Male,Suburban,Medicare Advantage,45.7,4,63.1,1,1,...,Q4_High,65-79,1,High,Q1_Low,1,3,Moderate,Attributed,1
4,M00005,38,Male,Suburban,Medicare Advantage,32.4,0,55.6,0,0,...,Q4_High,35-49,0,Low,Q1_Low,0,0,NaN,Not Attributed,0


## 3. Define target and features

In [3]:
target = "monthly_cost"  # Define the regression target variable

drop_cols = [
    "member_id",  # Identifier, not a predictive feature
    target,  # Target variable must be removed from predictors
    "high_cost_member",  # Derived from monthly_cost, so this leaks target information
    "awv_completed"  # Exclude AWV outcome to keep cost prediction focused on risk, access, and utilization
]

X = df.drop(columns=drop_cols, errors="ignore")  # Create predictors after removing leakage-prone columns
y = df[target]  # Store monthly cost as the regression target

numeric_cols = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]  # Select numeric predictors for scaling
).columns.tolist()  # Store numeric column names as a list

categorical_cols = X.select_dtypes(
    include=["object", "string", "category", "bool"]  # Select categorical/text-like predictors for encoding
).columns.tolist()  # Store categorical column names as a list

## 4. Train-test split


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # splitting data into training and testing sets (80% train, 20% test)

## 5. Build preprocessing

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),  # Standardize numeric predictors
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)  # One-hot encode categorical predictors
    ]
)

## 6. Build baseline models

In [6]:
linear_model = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression()) # using default Linear Regression without regularization for baseline comparison
])

ridge_pipeline = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("regressor", Ridge()) # using default alpha value for Ridge regression
])

lasso_pipeline = Pipeline(steps = [
    ("preprocessor", preprocessor),
    ("regressor", Lasso(max_iter = 10000)) # increasing max_iter to ensure convergence for Lasso regression
])

## 7. Fit the plain linear regression baseline

In [7]:
linear_model.fit(X_train, y_train)  # fitting linear regression model on training data

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

## 8. Set tuning grids

In [8]:
ridge_param_grid = {
    "regressor__alpha" : [0.001, 0.01, 0.1, 1.0, 10.0, 100.0] # defining a range of alpha values for Ridge regression to tune
} # Ridge can handle a wider range of alpha values, including larger ones, compared to Lasso which typically requires smaller alpha values to avoid over-penalization and feature elimination.

lasso_param_grid = {
    "regressor__alpha" : [0.001, 0.01, 0.1, 1.0, 10.0]
} # defining a range of alpha values for Lasso regression to tune (Lasso typically requires smaller alpha values than Ridge)


# Why this grid:
# it covers weak to strong penalties for both Ridge and Lasso, allowing us to find the optimal level of regularization for our dataset. The range includes very small alpha values (0.001, 0.01) that provide minimal regularization, moderate values (0.1, 1.0) that offer a balance between bias and variance, and larger values (10.0, 100.0 for Ridge) that impose stronger penalties to prevent overfitting. This comprehensive grid ensures we can identify the best alpha for both models based on their performance on the validation set.
# it is simple enough for learning
# it avoids fake precision


## 9. Tune Ridge with cross-validation

In [9]:
ridge_search = GridSearchCV(
    estimator=ridge_pipeline,  # Pipeline containing preprocessing and Ridge model
    param_grid=ridge_param_grid,  # Ridge alpha values to test
    cv=5,  # Use 5-fold cross-validation
    scoring="neg_root_mean_squared_error",  # Tune based on RMSE
    n_jobs=-1  # Use all available cores
)

ridge_search.fit(X_train, y_train)  # Fit Ridge search on training data

print("Best Ridge alpha:", ridge_search.best_params_) 
print("Best Ridge CV score:", ridge_search.best_score_)
print("Best Ridge CV RMSE:", -ridge_search.best_score_)  # Convert negative RMSE back to positive RMSE

Best Ridge alpha: {'regressor__alpha': 10.0}
Best Ridge CV score: -524.7318154241946
Best Ridge CV RMSE: 524.7318154241946


## 10. Tune Lasso with cross-validation

In [10]:
lasso_search = GridSearchCV(estimator = lasso_pipeline,
                            param_grid = lasso_param_grid, 
                            cv = 5, 
                            scoring = "neg_root_mean_squared_error",
                            n_jobs = -1  ) # Set up GridSearchCV for Lasso regression using 5-fold cross-validation and negative RMSE as the scoring metric
lasso_search.fit(X_train, y_train)

print("Best Lasso alpha:", lasso_search.best_params_)
print("Best Lasso CV score:", lasso_search.best_score_)
print("Best Lasso CV RMSE:", -lasso_search.best_score_)  # Convert negative RMSE back to positive RMSE

Best Lasso alpha: {'regressor__alpha': 1.0}
Best Lasso CV score: -524.1154981364562
Best Lasso CV RMSE: 524.1154981364562


## 11. Evaluate all final models on test set

In [11]:
def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test) # making predictions on the test set
    mae = mean_absolute_error(y_test, y_pred) # calculating Mean Absolute Error
    mse = mean_squared_error(y_test, y_pred) # calculating Mean Squared Error
    rmse = np.sqrt(mse) # calculating Root Mean Squared Error
    r2 = r2_score(y_test, y_pred) # calculating R-squared score

    return pd.DataFrame({
        "model" : [model_name],
        "MAE" : [mae],
        "RMSE" : [rmse],
        "R2" : [r2]
    })

results_df = pd.concat([
    evaluate_model(linear_model, X_test, y_test, "Linear Regression"),
    evaluate_model(ridge_search.best_estimator_, X_test, y_test, "Tuned Ridge"),
    evaluate_model(lasso_search.best_estimator_, X_test, y_test, "Tuned Lasso")
], ignore_index = True)

results_df 

,model,MAE,RMSE,R2
0,Linear Regression,363.402801,635.784246,0.777007
1,Tuned Ridge,363.072248,634.660630,0.777795
2,Tuned Lasso,362.531760,634.334303,0.778023


## 12. Extract best tuned coefficients


In [12]:
feature_names = ridge_search.best_estimator_.named_steps["preprocessor"].get_feature_names_out() # getting feature names after preprocessing (scaling and encoding)

ridge_best_coefs = ridge_search.best_estimator_.named_steps["regressor"].coef_
lasso_best_coefs = lasso_search.best_estimator_.named_steps["regressor"].coef_
coef_df = pd.DataFrame({
    "feature" : feature_names,
    "ridge_coef" : ridge_best_coefs,
    "lasso_coef" : lasso_best_coefs
})

coef_df["ridge_abs"] = np.abs(coef_df["ridge_coef"]) # calculating absolute values of Ridge coefficients to compare feature importance without regard to direction (positive or negative)
coef_df["lasso_abs"] = np.abs(coef_df["lasso_coef"]) # calculating absolute values of Lasso coefficients to compare feature importance without regard to direction (positive or negative)

coef_df.sort_values("ridge_abs", ascending = False).head(20)

,feature,ridge_coef,lasso_coef,ridge_abs,lasso_abs
25,cat__chronic_burden_group_Moderate,-614.675444,-649.508083,614.675444,649.508083
2,num__chronic_condition_count,558.819458,545.084713,558.819458,545.084713
15,cat__plan_type_Medicaid,-527.079235,-543.174353,527.079235,543.174353
8,num__ip_admits,457.446044,422.937669,457.446044,422.937669
16,cat__plan_type_Medicare Advantage,-378.874729,-392.719621,378.874729,392.719621
24,cat__chronic_burden_group_Low,-373.650325,-426.374717,373.650325,426.374717
10,num__total_acute_visits,263.346298,335.127470,263.346298,335.127470
17,cat__engagement_group_Q2,-132.838235,-119.545724,132.838235,119.545724
18,cat__engagement_group_Q3,-112.854450,-92.514416,112.854450,92.514416
14,cat__region_Urban,-80.629342,-72.931757,80.629342,72.931757


In [13]:
coef_df.sort_values(
    "lasso_abs",  # Sort by Lasso coefficient magnitude
    ascending=False  # Show largest absolute Lasso coefficients first
).head(20)  # Display strongest tuned Lasso coefficients

,feature,ridge_coef,lasso_coef,ridge_abs,lasso_abs
25,cat__chronic_burden_group_Moderate,-614.675444,-649.508083,614.675444,649.508083
2,num__chronic_condition_count,558.819458,545.084713,558.819458,545.084713
15,cat__plan_type_Medicaid,-527.079235,-543.174353,527.079235,543.174353
24,cat__chronic_burden_group_Low,-373.650325,-426.374717,373.650325,426.374717
8,num__ip_admits,457.446044,422.937669,457.446044,422.937669
16,cat__plan_type_Medicare Advantage,-378.874729,-392.719621,378.874729,392.719621
10,num__total_acute_visits,263.346298,335.127470,263.346298,335.127470
17,cat__engagement_group_Q2,-132.838235,-119.545724,132.838235,119.545724
18,cat__engagement_group_Q3,-112.854450,-92.514416,112.854450,92.514416
0,num__age,74.839407,87.000852,74.839407,87.000852


## 13. Check Lasso sparsity

In [14]:
num_lasso_zero = (coef_df["lasso_coef"] == 0).sum()  # Count coefficients set exactly to zero by Lasso
total_features = len(coef_df)  # Count total transformed features
percent_zeroed = num_lasso_zero / total_features  # Calculate share of zeroed coefficients

print("Lasso coefficients equal to zero:", num_lasso_zero)  # Print number of zero coefficients
print("Total features:", total_features)  # Print total feature count
print("Percent zeroed:", percent_zeroed)  # Print share of coefficients set to zero


Lasso coefficients equal to zero: 8
Total features: 33
Percent zeroed: 0.24242424242424243


## 14. Findings

The purpose of this notebook was to tune Ridge and Lasso regularized regression models for predicting monthly member cost.

The target variable was `monthly_cost`.

The predictor set excluded `member_id`, `high_cost_member`, and `awv_completed`. `member_id` was excluded because it is only an identifier. `high_cost_member` was excluded because it is derived from `monthly_cost` and would create target leakage. `awv_completed` was excluded to keep the cost prediction model focused on member characteristics, risk, access, and utilization.

Ridge and Lasso were tuned using 5-fold cross-validation rather than manually chosen alpha values.

Both models were tuned using negative root mean squared error, so the selected alpha values were based on estimated RMSE performance during cross-validation.

The tuned models were then evaluated on the held-out test set and compared against plain Linear Regression using MAE, RMSE, and R².

If tuned Ridge or Lasso outperformed plain Linear Regression, that suggests regularization improved generalization for this synthetic dataset. If performance was similar across models, that suggests the baseline linear model was already reasonably stable.

Lasso coefficients equal to zero should be interpreted as model-specific sparsity under the selected penalty, not proof that those predictors are universally irrelevant.

Because this is synthetic data, the observed relationships reflect the assumptions built into the data-generation process and should not be interpreted as real-world causal evidence.